# Triaxial compression, TWO-PISTON — single strain level

Analysis of **one** applied-strain level of a `triaxial_compression_two_pist` run: drained consolidation at
**constant bath pressure** — both solvent reservoirs are closed by NPT-pistons (Marioni et al., J. Membr. Sci.
738 (2026) 124837, Eq. 3) held at `P_target`, and a solvent-transparent **dry piston** loads the network through
the same cumulative strain sweep as `triaxial_compression`.  Set `LEVEL` and the run identifiers in **Config**,
run the **sync** cell once, then run everything.  All analysis code lives in `scripts/lib/triaxial.py`; this
notebook only configures it and draws the eleven one-piston figures **plus two two-piston panels**: the wet-piston
bath check (P_feed, P_perm measured vs `P_target` over the hold) and the solvent expelled (wet-piston
displacement).  The dry piston plays the role of "the piston" everywhere (its force is the network load; the
piston files carry it in the first value column, `[dry | feed | perm]`).

Method notes and the file list are in **Notes** at the end (2026-09-16).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
tri = importlib.reload(tri)            # pick up edits to lib/triaxial.py without a kernel restart
tri.setup_style()
print('analysis code: ', LIB / 'triaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
LEVEL = "0.10"      # the ONE applied-strain level analysed here (one of STRAIN_TARGETS in the .batch; files _c<LEVEL>)
cfg = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # <steps> tag = each level's auto-sized HOLD LENGTH; None resolves it from the files
    RUN_ID      = "two_pist_comp_1",  # local folder under flow_data_local/{compression,plots}
    COMP_LEVELS = [LEVEL],
    mode        = "compression",  # two-piston COMPRESSION run ...
    two_pist    = True,           # ... in lammps_runs/triaxial_compression_two_pist on Expanse; syncs the wet-piston files
    # ---- measurement windows ----
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    # ---- solvent volume fractions (lib/volfrac.py) ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    # ---- M and G as increments from the eps = 0 reference ----
    M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    # ---- D_c consolidation fit + hold-adequacy check ----
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
)
# Every other knob keeps its default -- see `tri.Config` in lib/triaxial.py.  P_BARO (= P_target of the
# wet pistons, 1.5) normalises the total-stress panels and is the reference line of the bath check.

In [ ]:
# Pull the files this notebook reads from Expanse in ONE login (password + TOTP prompts).
# two_pist=True adds piston_pressure / permeation / pressure_reservoirs to the file list.
SYNC, FORCE_SYNC = True, False
if SYNC:
    tri.sync_from_expanse(cfg, levels=[LEVEL], force=FORCE_SYNC)

In [ ]:
# Load + compute everything:
#   R  -- the eps = 0 reference state (also the zero-strain bath pressures on the wet pistons)
#   L  -- this level: stresses, Terzaghi split, dry-piston plateau, M, G, D_c, kappa, + L['wet'] (bath check)
R = tri.load_reference(cfg)
L = tri.load_level(cfg, R, LEVEL)
assert L is not None, f'level {LEVEL}: core files missing -- run the sync cell'
tri.add_volume_fractions(cfg, R, [L])
tri.print_summary(cfg, [L])

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic -- solid ε_Rg, dashed ε_BB, faint ε_piston (dry piston vs support); shaded = plateau window
tri.fig_strain(cfg, R, [L]);

In [ ]:
# 2 · Solvent volume fraction φ_s: mass fraction, Voronoi and λ-calibrated Voronoi -- reference (ε = 0) vs compressed
tri.fig_volfrac(cfg, R, L);

In [ ]:
# 3 · Total stress evolution σ^t_zz, σ^t_xx, σ^t_yy / P_bath -- reference (dashed) -> hold (cividis) -> plateau (bold).
#     Pore baseline = the FEED reservoir interior (the box top is vacuum in the two-piston geometry)
tri.fig_total_stress(cfg, R, L);

In [ ]:
# 4 · Solvent and polymer partial σ_zz evolutions with the total superimposed
tri.fig_partial_stress(cfg, R, L);

In [ ]:
# 5 · Network stress evolution σ'_zz, σ'_xx, σ'_yy (Terzaghi: σ' = σ^t − p_pore), 95 % bands
tri.fig_network_stress(cfg, R, L);

In [ ]:
# 6 · DRY-piston pressure P = F_z/A vs step, linear + log; green = auto-selected plateau window
tri.fig_piston(cfg, R, L);

In [ ]:
# 7 · Longitudinal modulus M (increments from ε = 0): network vs dry piston
tri.fig_M(cfg, R, L);

In [ ]:
# 8 · Network-stress anisotropy σ'_zz/σ'_xx and σ'_zz/σ'_yy vs step  (= M/(M − 2G))
tri.fig_ratio(cfg, R, L);

In [ ]:
# 9 · Shear modulus G = (σ'_zz − σ'_ii)/(2ε) from xx and from yy
tri.fig_G(cfg, R, L);

In [ ]:
# 10 · Cooperative diffusivity D_c: consolidation fit of u_z(ζ, t)/L
tri.fig_Dc(cfg, R, L);

In [ ]:
# 11 · κ = D_c/M (hydraulic permeability / viscosity) from the network and the piston M
tri.fig_kappa(cfg, R, L);

In [ ]:
# 12 · TWO-PISTON: bath check -- P_feed and P_perm measured on the wet pistons vs P_target over the hold (+ P_dry)
tri.fig_wet_pistons(cfg, R, L);

In [ ]:
# 13 · TWO-PISTON: solvent expelled = A·(feed-piston rise + permeate-piston descent) vs time
tri.fig_solvent_expelled(cfg, R, L);


## Notes

Everything below is reference material — nothing above depends on reading it.

### Two-piston specifics (2026-09-16)

* **Geometry**: `zlo` = vacuum | permeate piston (type 6) | permeate reservoir | support (type 4) | gel | feed
  reservoir with the **dry piston** (type 7) | feed piston (type 5) | vacuum = `zhi`.  `lb/triaxial.py` reads the
  four wall planes from `traj_ref`; the dry piston is `R['z_piston']` (the loading plate), the wet pistons are
  `R['z_feed']`, `R['z_perm']`.
* **Pore baseline**: the far-reservoir window used for `p_pore` is the **feed reservoir interior**
  `[gel top + 2 bins, feed piston − 1.5 σ]`, not `z/Lz ≈ 0.95` (that is vacuum here).  `R['bw']` shows the bins.
* **Piston files** carry one column set per piston, `[dry | feed | perm]`, with a `# step …` header.  The
  compression loaders read the first value column (dry piston = network load), so `M_piston`, `P_ref`, the
  plateau window etc. mean exactly what they do for one-piston runs.  `L['wet']` holds the wet-piston pressures
  (`piston_pressure_…_c<lvl>.dat`) and the solvent expelled (`permeation_…_c<lvl>.dat`, `dV_total`).
* **Bath check**: figure 12 must show `P_feed`, `P_perm` ≈ `P_target` throughout the hold (the pistons regulate
  the reservoirs; a systematic offset means the settle was too short or the damping too strong — see the
  `C_crit` / period printout in the `.lmp` log).  Figure 13 shows where the expelled solvent went; the feed
  piston rise must stay below the converter's `margin_feed`.
* **Files read** (into `flow_data_local/compression/<RUN_ID>/`): as `triaxial_compression_single.ipynb` plus
  `piston_pressure[_c<lvl>]`, `permeation_c<lvl>`, `pressure_reservoirs[_ref]`; `piston_force_avg_ref` now has
  three columns.

### Everything else

The one-piston Notes (strain diagnostic, Terzaghi split, plateau window, M, G, D_c, κ, volume fractions) apply
unchanged — see `triaxial_compression_single.ipynb`.